# MeMoMe model inspection

This notebook is intended for lightweight inspection of MeMoMe input and merged SBML models.

Note: Jupyter uses a *single* Python environment per kernel. If `cobra` is not available, switch the notebook kernel to an environment where `cobra` is installed (e.g., the project’s FVA/analysis environment).

In [72]:
import sys
import platform

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())

try:
    import cobra  # noqa: F401
    print("cobra: OK")
except Exception as e:
    print("cobra: NOT AVAILABLE (switch kernel to an env with cobrapy)")
    print(" ", type(e).__name__, e)


Python: 3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]
Executable: /home/steflor/projects/MeMoMe/.venv-fva/bin/python
Platform: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
cobra: OK


In [73]:
from pathlib import Path

ROOT = Path.cwd()

PATHS = {
    # MeMoMe inputs
    "input_host_model": ROOT / "tests/dat/manually_merged_models/gapseq_recon3D/M1_recon3D_301_modified.xml",
    "input_microbiome_model": ROOT / "tests/dat/manually_merged_models/gapseq_recon3D/M2_bacterial_model.xml",

    # Merged models used in comparisons
    "merged_auto_memome": ROOT / "tests/dat/manually_merged_models/gapseq_recon3D/output/automatically_merged_metamodel.xml",
    "merged_manual": ROOT / "tests/dat/manually_merged_models/gapseq_recon3D/output/merged_model_2025_prefixed_normalized_diet_fixIEX.xml",
}

for k, p in PATHS.items():
    print(f"{k}:\n  {p}\n  exists={p.exists()}\n")


input_host_model:
  /home/steflor/projects/MeMoMe/tests/dat/manually_merged_models/gapseq_recon3D/M1_recon3D_301_modified.xml
  exists=True

input_microbiome_model:
  /home/steflor/projects/MeMoMe/tests/dat/manually_merged_models/gapseq_recon3D/M2_bacterial_model.xml
  exists=True

merged_auto_memome:
  /home/steflor/projects/MeMoMe/tests/dat/manually_merged_models/gapseq_recon3D/output/automatically_merged_metamodel.xml
  exists=True

merged_manual:
  /home/steflor/projects/MeMoMe/tests/dat/manually_merged_models/gapseq_recon3D/output/merged_model_2025_prefixed_normalized_diet_fixIEX.xml
  exists=True



In [74]:
input_host_model = cobra.io.read_sbml_model(PATHS["input_host_model"])
input_host_model

Name,COBRAModel
Memory address,7293e03dc3e0
Number of metabolites,5835
Number of reactions,10604
Number of genes,2248
Number of groups,103
Objective expression,1.0*biomass_maintenance - 1.0*biomass_maintenance_reverse_95d2f
Compartments,"Cytoplasm, Lysosome, Mitochondrion, Endoplasmic_reticulum, Extracellular, Peroxisome, Nucleus, Golgi, unknownCompartment4"


In [75]:
input_bacterial_model = cobra.io.read_sbml_model(PATHS["input_microbiome_model"])
input_bacterial_model

No objective coefficients in model. Unclear what should be optimized


Name,gapseq_metamodel
Memory address,7293db26e9f0
Number of metabolites,3142
Number of reactions,4370
Number of genes,0
Number of groups,0
Objective expression,0
Compartments,"c, e, p"


In [76]:
manually_merged_model = cobra.io.read_sbml_model(PATHS["merged_manual"])
manually_merged_model

No objective coefficients in model. Unclear what should be optimized


Name,human_with_mouse_microbiome_metamodel
Memory address,7293d8adf590
Number of metabolites,10622
Number of reactions,16619
Number of genes,2212
Number of groups,103
Objective expression,0
Compartments,"Cytoplasm, Extracellular (new common compartment), Peroxisome, Mitochondrion, Golgi, Lysosome, Endoplasmic_reticulum, Nucleus, unknownCompartment4, Bacterial_cytoplasm, Periplasm, Feeding (new external)"


In [77]:
automatically_merged_model = cobra.io.read_sbml_model(PATHS["merged_auto_memome"])
automatically_merged_model

Name,merged_model
Memory address,7293d75db560
Number of metabolites,10638
Number of reactions,16635
Number of genes,2212
Number of groups,103
Objective expression,1.0*model1_biomass_maintenance - 1.0*model1_biomass_maintenance_reverse_75b22
Compartments,"c, l, m, r, e, x, n, g, i, p, t"


In [78]:
# Count how many metabolites are effectively "matched" (i.e., connected host↔microbe) in each merged model.
#
# Logic used:
# - Manual merge: a metabolite pool is counted as matched if BOTH a host connector reaction (H_IEX_*) and a
#   microbial connector reaction (M_IEX_*) feed the SAME metabolite pool in compartment "f".
#   If an IEX reaction feeds both a metabolite and a proton (h[f]) in "f", the proton is ignored and the
#   non-proton metabolite is used. Proton-only IEX reactions (only h[f]) are counted as the proton match.
# - MeMoMe merge: a metabolite is counted as matched if its translation-compartment metabolite (*_t in
#   compartment "t") participates in TR reactions that connect it to BOTH model1_* and model2_* metabolites.

from src.handle_metabolites_prefix_suffix import handle_metabolites_prefix_suffix


def _base_id(met_id: str) -> str | None:
    try:
        return handle_metabolites_prefix_suffix(met_id)
    except Exception:
        return None


def _is_proton_feed_met(met) -> bool:
    if getattr(met, "compartment", None) != "f":
        return False
    return _base_id(met.id) == "h" or (met.name or "").lower() in {"proton", "h+"}


def _feed_bases_from_iex(model, rxn_prefix: str) -> tuple[set[str], list[str]]:
    bases: set[str] = set()
    skipped_rxns: list[str] = []
    for rxn in model.reactions:
        if not rxn.id.startswith(rxn_prefix):
            continue

        feed_f = [m for m in rxn.metabolites if m.compartment == "f"]
        if len(feed_f) == 0:
            skipped_rxns.append(rxn.id)
            continue

        feed_non_proton = [m for m in feed_f if not _is_proton_feed_met(m)]

        # Most IEX reactions have exactly one non-proton metabolite in the feeding compartment.
        if len(feed_non_proton) > 0:
            b = _base_id(feed_non_proton[0].id)
            if b:
                bases.add(b)
            continue

        # Proton-only IEX: count as a real match for the proton pool.
        feed_proton = [m for m in feed_f if _is_proton_feed_met(m)]
        if len(feed_proton) == 0:
            skipped_rxns.append(rxn.id)
            continue

        b = _base_id(feed_proton[0].id)
        if b:
            bases.add(b)

    return bases, skipped_rxns


# Manual merge matches (via feeding compartment "f")
host_feed_bases, host_skipped = _feed_bases_from_iex(manually_merged_model, "H_IEX_")
micro_feed_bases, micro_skipped = _feed_bases_from_iex(manually_merged_model, "M_IEX_")
matched_manual = host_feed_bases & micro_feed_bases


# MeMoMe merge matches (via translation compartment "t")
t_mets = [m for m in automatically_merged_model.metabolites if m.compartment == "t"]
matched_auto: set[str] = set()
for tm in t_mets:
    b = _base_id(tm.id)
    if not b:
        continue
    sides: set[str] = set()
    for rxn in tm.reactions:
        if not rxn.id.startswith("TR_"):
            continue
        for other in rxn.metabolites:
            if other.id == tm.id:
                continue
            if other.id.startswith("model1_"):
                sides.add("model1")
            elif other.id.startswith("model2_"):
                sides.add("model2")
    if sides == {"model1", "model2"}:
        matched_auto.add(b)


print("Matched metabolite pools (manual merge):", len(matched_manual))
print("  includes proton (h):", "h" in matched_manual)
print("  host feed pools:", len(host_feed_bases), "| microbe feed pools:", len(micro_feed_bases))
print("  skipped IEX (no feeding pool in compartment f):", len(set(host_skipped + micro_skipped)))
print()
print("Matched translation metabolites (MeMoMe merge):", len(matched_auto))
print("  includes proton (h):", "h" in matched_auto)
print("  total translation metabolites (compartment t):", len(t_mets))


Matched metabolite pools (manual merge): 177
  includes proton (h): True
  host feed pools: 1561 | microbe feed pools: 261
  skipped IEX (no feeding pool in compartment f): 0

Matched translation metabolites (MeMoMe merge): 168
  includes proton (h): True
  total translation metabolites (compartment t): 1661


In [79]:
# --- Overlap of metabolites and reactions (manual vs MeMoMe) ---
#
# We compare the two merged models after canonicalizing IDs to account for:
# - organism/model prefixes (H_/M_ vs model1_/model2_)
# - compartment suffixes ([c], (c), _c, __91__c__93__, *_t, etc.)
# - connector conventions (manual IEX_* vs MeMoMe TR_*)
#
# Canonicalization below mirrors the logic used in:
# tests/dat/manually_merged_models/gapseq_recon3D/output/compare_models_ids/00_compare_models_ids.py

import re
from src.handle_metabolites_prefix_suffix import handle_metabolites_prefix_suffix

SPECIAL_PREFIXES = ("EX_", "DM_", "SK_", "sink_")


def strip_reaction_compartment(rxn_id: str) -> str:
    rid = rxn_id
    if rid.endswith("_t"):
        rid = rid[:-2]
    rid = re.sub(r"\([A-Za-z0-9_]+\)$", "", rid)
    rid = re.sub(r"\[[A-Za-z0-9_]+\]$", "", rid)
    rid = re.sub(r"_[a-z]\d?$", "", rid)
    return rid


def strip_model_prefix(value: str) -> str:
    for prefix in ("model1_", "model2_", "H_", "M_"):
        if value.startswith(prefix):
            return value[len(prefix) :]
    return value


def canonical_met_id(met_id: str) -> str:
    base = strip_model_prefix(met_id)
    cleaned = handle_metabolites_prefix_suffix(base)
    return cleaned if cleaned is not None else base


def canonical_reaction_id(rxn_id: str) -> str:
    rid = rxn_id

    # MeMoMe translation reactions: TR_model{1,2}_*  -> TR_*
    if rid.startswith("TR_model1_") or rid.startswith("TR_model2_"):
        rid = "TR_" + rid.split("_", 2)[2]

    # Translation / connector reactions across models -> TR_*
    if rid.startswith("TR_"):
        base = strip_model_prefix(rid[len("TR_") :])
        return "TR_" + strip_reaction_compartment(base)

    # Manual connectors: H_IEX_*/M_IEX_* (and any plain IEX_*) -> TR_*
    if rid.startswith(("IEX_", "H_IEX_", "M_IEX_")):
        if rid.startswith("IEX_"):
            base = strip_model_prefix(rid[len("IEX_") :])
        else:
            base = strip_model_prefix(rid.split("IEX_", 1)[1])
        return "TR_" + strip_reaction_compartment(base)

    stripped = strip_model_prefix(rid)
    if stripped.startswith("IEX_"):
        return "TR_" + strip_reaction_compartment(stripped[len("IEX_") :])

    # DM_/SK_/sink_ keep their prefix but drop model/compartment decorations
    for prefix in SPECIAL_PREFIXES:
        if rid.startswith(prefix):
            base = strip_model_prefix(rid[len(prefix) :])
            return prefix + strip_reaction_compartment(base)

    # Exchange reactions: EX_ / H_EX_ / M_EX_ / EX_model{1,2}_* -> EX_*
    for ex_prefix in ("EX_", "H_EX_", "M_EX_"):
        if rid.startswith(ex_prefix):
            base = rid[len(ex_prefix) :]
            base = strip_model_prefix(base)
            return "EX_" + strip_reaction_compartment(base)

    if rid.startswith("EX_model1_") or rid.startswith("EX_model2_"):
        base = rid.split("_", 2)[2]
        return "EX_" + strip_reaction_compartment(base)

    # Fallback: strip organism/model prefix + compartment
    rid = strip_model_prefix(rid)
    for prefix in SPECIAL_PREFIXES:
        if rid.startswith(prefix):
            return prefix + strip_reaction_compartment(rid[len(prefix) :])

    return strip_reaction_compartment(rid)


# Canonicalized metabolite sets
manual_mets = {canonical_met_id(m.id) for m in manually_merged_model.metabolites}
auto_mets = {canonical_met_id(m.id) for m in automatically_merged_model.metabolites}

met_intersection = manual_mets & auto_mets
met_only_manual = manual_mets - auto_mets
met_only_auto = auto_mets - manual_mets

print("Metabolites (canonicalized, compartment/prefix-agnostic):")
print("  manual:", len(manual_mets))
print("  auto:", len(auto_mets))
print("  intersection:", len(met_intersection))
print("  only manual:", len(met_only_manual))
print("  only auto:", len(met_only_auto))
print("  examples only manual:", sorted(list(met_only_manual))[:15])
print("  examples only auto:", sorted(list(met_only_auto))[:15])
print()

# Canonicalized reaction sets
manual_rxns = {canonical_reaction_id(r.id) for r in manually_merged_model.reactions}
auto_rxns = {canonical_reaction_id(r.id) for r in automatically_merged_model.reactions}

rxn_intersection = manual_rxns & auto_rxns
rxn_only_manual = manual_rxns - auto_rxns
rxn_only_auto = auto_rxns - manual_rxns

print("Reactions (canonicalized, prefix-agnostic; IEX/TR normalized; EX normalized):")
print("  manual:", len(manual_rxns))
print("  auto:", len(auto_rxns))
print("  intersection:", len(rxn_intersection))
print("  only manual:", len(rxn_only_manual))
print("  only auto:", len(rxn_only_auto))
print("  examples only manual:", sorted(list(rxn_only_manual))[:15])
print("  examples only auto:", sorted(list(rxn_only_auto))[:15])


Metabolites (canonicalized, compartment/prefix-agnostic):
  manual: 5667
  auto: 5667
  intersection: 5667
  only manual: 0
  only auto: 0
  examples only manual: []
  examples only auto: []

Reactions (canonicalized, prefix-agnostic; IEX/TR normalized; EX normalized):
  manual: 16600
  auto: 16616
  intersection: 16588
  only manual: 12
  only auto: 28
  examples only manual: ['EX_4hpro', 'EX_CE1557', 'EX_CE4843', 'EX_HC00229', 'EX_HC01610', 'EX_cpd00453', 'EX_cpd01376', 'EX_cpd02992', 'EX_ddecrn', 'EX_oxyp1rb', 'EX_oxyp7rb', 'EX_sfcys']
  examples only auto: ['EX_4hpro_LT', 'EX_CE1556', 'EX_M02447', 'EX_M02451', 'EX_cpd00027', 'EX_cpd00105', 'EX_cpd00154', 'EX_cpd00221', 'EX_cpd00224', 'EX_cpd00644', 'EX_cpd00751', 'EX_cpd00949', 'EX_cpd01399', 'EX_cpd01861', 'EX_cpd02829']


In [80]:
host_skipped

[]

In [81]:
micro_skipped

[]

In [82]:
len(matched_auto.difference(matched_manual))


3

In [103]:
matched_auto.difference(matched_manual)

{'core4', 'elaid', 'oh1'}

In [83]:
len(matched_manual.difference(matched_auto))

12

In [84]:
len(matched_auto.intersection(matched_manual))

165

In [85]:
len(input_host_model.exchanges)

1563

In [86]:
len([rxn for rxn in input_host_model.reactions if rxn.id.startswith("EX_")])

1563

In [87]:
len(input_bacterial_model.exchanges)

261

In [88]:
exchanges_bact = [rxn for rxn in input_bacterial_model.reactions if rxn.id.startswith("EX_")]
len(exchanges_bact)

263

In [89]:
[ex.id for ex in exchanges_bact if "EX_Biomass_" in ex.id]

['EX_Biomass_Gram_negative[c]', 'EX_Biomass_Gram_positive[c]']

In [90]:
#check if "Biomass_Gram_negative" is substring of any reaction id of bacterial model
[reac.id for reac in input_bacterial_model.reactions if "iomass" in reac.id]


['Bacterial_Gram_negative_biomass_reaction',
 'Bacterial_Gram_positive_biomass_reaction',
 'EX_Biomass_Gram_negative[c]',
 'EX_Biomass_Gram_positive[c]']

In [91]:
len(manually_merged_model.boundary)

1895

In [92]:
exchanges_manual = [rxn for rxn in manually_merged_model.reactions if rxn.id.startswith("EX_")]
len(exchanges_manual)

1647

In [93]:
len(automatically_merged_model.boundary)

1909

In [94]:
exchanges_auto = [rxn for rxn in automatically_merged_model.reactions if rxn.id.startswith("EX_")]
len(exchanges_auto)

1661

In [95]:
diet_bact = [rxn for rxn in input_bacterial_model.reactions if rxn.id.startswith("EX_") and rxn.lower_bound < 0]
len(diet_bact)

88

In [96]:
diet_host = [rxn for rxn in input_host_model.reactions if rxn.id.startswith("EX_") and rxn.lower_bound < 0]
len(diet_host)

97

In [97]:
diet_manual = [rxn for rxn in manually_merged_model.reactions if rxn.id.startswith("EX_") and rxn.lower_bound < 0]
len(diet_manual)

110

In [98]:
diet_auto = [rxn for rxn in automatically_merged_model.reactions if rxn.id.startswith("EX_") and rxn.lower_bound < 0]
len(diet_auto)

114

In [102]:
set(diet_manual)

{<Reaction EX_adn[f] at 0x7293d2392300>,
 <Reaction EX_ala_L[f] at 0x7293d23eeb40>,
 <Reaction EX_arab_L[f] at 0x7293d224a600>,
 <Reaction EX_arach[f] at 0x7293d2837680>,
 <Reaction EX_arachd[f] at 0x7293d2837770>,
 <Reaction EX_arg_L[f] at 0x7293d23ef440>,
 <Reaction EX_ascb_L[f] at 0x7293d2837800>,
 <Reaction EX_asn_L[f] at 0x7293d2392600>,
 <Reaction EX_asp_L[f] at 0x7293d23926c0>,
 <Reaction EX_avite1[f] at 0x7293d2837b60>,
 <Reaction EX_btn[f] at 0x7293d224ac00>,
 <Reaction EX_but[f] at 0x7293d2392840>,
 <Reaction EX_ca2[f] at 0x7293d20b8f80>,
 <Reaction EX_caro[f] at 0x7293d28983b0>,
 <Reaction EX_chol[f] at 0x7293d28985c0>,
 <Reaction EX_chsterol[f] at 0x7293d28988f0>,
 <Reaction EX_cl[f] at 0x7293d20b9100>,
 <Reaction EX_clpnd[f] at 0x7293d2898a70>,
 <Reaction EX_cpd00030[f] at 0x7293d20bbb30>,
 <Reaction EX_cpd00058[f] at 0x7293d20b9790>,
 <Reaction EX_cpd00149[f] at 0x7293d20b9220>,
 <Reaction EX_cpd00166[f] at 0x7293d1f187a0>,
 <Reaction EX_cpd00254[f] at 0x7293d20bbc20>,
 <